In [1]:
# ===== CELL 0: setup =====
import os, sys, json, glob, random
from pathlib import Path
import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu  :", torch.cuda.get_device_name(0))

WORK = "/kaggle/working/reid_v2"
os.makedirs(WORK, exist_ok=True)

hits = glob.glob("/kaggle/input/**/split.json", recursive=True)
assert hits, "split.json nahi mila -> phase4_Re_id ka notebook output attach karo"
OLD_SPLIT = hits[0]
print("\npurana split :", OLD_SPLIT)

ROOTS = []
for dirpath, dirnames, _ in os.walk("/kaggle/input"):
    if any(d.lower().startswith("cam_") for d in dirnames):
        ROOTS.append(dirpath)
ROOTS = sorted(set(ROOTS))
print("\ncrop roots mile:", len(ROOTS))
for r in ROOTS:
    n = len([d for d in os.listdir(r) if d.lower().startswith("cam_")])
    print(f"  {r}  ({n} chunks)")

assert len(ROOTS) >= 2, "sirf ek root mila -> cam12-extra-crops attach karo"

torch: 2.10.0+cu128 | cuda: True
gpu  : Tesla T4

purana split : /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/split.json

crop roots mile: 2
  /kaggle/input/datasets/payal1238gupta/cam-12-chunk-0-12  (15 chunks)
  /kaggle/input/datasets/payal1238gupta/cam12-extra-crops  (5 chunks)


In [2]:
# ===== CELL 1: multi-root scan + split =====
random.seed(42)
MIN_IMG = 5

old = json.load(open(OLD_SPLIT))
old_train_keys = set(old["train"])
eval_keys = {k for _, k in old["query"]}
print("purane train ids:", len(old_train_keys), "| purane eval ids:", len(eval_keys))

all_ids = {}
for root in ROOTS:
    tag = Path(root).name[:14]
    for ch in sorted(d for d in os.listdir(root) if d.lower().startswith("cam_")):
        if "chunk_032" in ch:
            continue
        for tid in os.listdir(os.path.join(root, ch)):
            p = os.path.join(root, ch, tid)
            if not os.path.isdir(p):
                continue
            imgs = sorted(glob.glob(os.path.join(p, "*.jpg")))
            if len(imgs) >= MIN_IMG:
                plain = f"{ch}/{tid}"
                key = plain if (plain in old_train_keys or plain in eval_keys) else f"{tag}/{plain}"
                all_ids[key] = imgs

print("kul identities (MIN_IMG=5):", len(all_ids))

train_map = {k: v for k, v in all_ids.items() if k not in eval_keys}
print("\ntrain ids :", len(train_map), f"({sum(len(v) for v in train_map.values())} images)")
print("purane se :", sum(1 for k in train_map if k in old_train_keys))
print("naye      :", sum(1 for k in train_map if k not in old_train_keys))

def fix(p):
    if os.path.exists(p):
        return p
    for r in ROOTS:
        cand = os.path.join(r, *p.split("/")[-3:])
        if os.path.exists(cand):
            return cand
    return p

query   = [(fix(p), k) for p, k in old["query"]]
gallery = [(fix(p), k) for p, k in old["gallery"]]
assert os.path.exists(query[0][0]), "eval image paths galat hain"
print("\neval (purana hi): query", len(query), "| gallery", len(gallery))

split = {"train": train_map, "query": query, "gallery": gallery}
json.dump(split, open(f"{WORK}/split_v2.json", "w"))

assert not (set(train_map) & eval_keys), "LEAK! train mein eval id hai"
print("leak check: OK")

purane train ids: 236 | purane eval ids: 58
kul identities (MIN_IMG=5): 757

train ids : 699 (57423 images)
purane se : 236
naye      : 463

eval (purana hi): query 58 | gallery 4924
leak check: OK


In [3]:
# ===== CELL 2: dataset + PK sampler =====
import torch, random
import numpy as np
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image

train_map  = split["train"]
train_keys = sorted(train_map)
key2label  = {k: i for i, k in enumerate(train_keys)}
NUM_CLASSES = len(train_keys)

H, W = 256, 128
P, K = 8, 4                      # 8 identities x 4 images = batch 32

train_tf = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.03),
    transforms.RandomGrayscale(p=0.1),
    transforms.Pad(10),
    transforms.RandomCrop((H, W)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), value=0),
])
eval_tf = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FlatReID(Dataset):
    """ek item = (image, label) — triplets ab batch ke andar bante hain"""
    def __init__(self, id_map, transform):
        self.items = [(p, key2label[k]) for k, v in id_map.items() for p in v]
        self.tf = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

class PKSampler(Sampler):
    """har batch mein P identities, har identity se K images"""
    def __init__(self, id_map, P, K):
        self.by_label = {}
        idx = 0
        for k, v in id_map.items():
            self.by_label.setdefault(key2label[k], []).extend(range(idx, idx + len(v)))
            idx += len(v)
        self.labels = list(self.by_label)
        self.P, self.K = P, K
        self.num_batches = len(self.labels) // P
    def __len__(self):
        return self.num_batches * self.P * self.K
    def __iter__(self):
        labs = self.labels[:]
        random.shuffle(labs)
        for b in range(self.num_batches):
            for lab in labs[b*self.P:(b+1)*self.P]:
                pool = self.by_label[lab]
                pick = random.sample(pool, self.K) if len(pool) >= self.K else \
                       [random.choice(pool) for _ in range(self.K)]
                for i in pick:
                    yield i

class EvalSet(Dataset):
    def __init__(self, pairs, transform):
        self.pairs, self.tf = pairs, transform
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        p, k = self.pairs[i]
        return self.tf(Image.open(p).convert("RGB")), k

train_ds = FlatReID(train_map, train_tf)
train_dl = DataLoader(train_ds, batch_size=P*K, sampler=PKSampler(train_map, P, K),
                      num_workers=4, drop_last=True, pin_memory=True, persistent_workers=True)
query_dl   = DataLoader(EvalSet(split["query"],   eval_tf), batch_size=64, num_workers=2)
gallery_dl = DataLoader(EvalSet(split["gallery"], eval_tf), batch_size=64, num_workers=2)

print("train identities :", NUM_CLASSES)
print("train images     :", len(train_ds))
print("batches/epoch    :", len(train_dl))
print("query / gallery  :", len(split["query"]), "/", len(split["gallery"]))
x, y = next(iter(train_dl))
print("batch:", tuple(x.shape), "| unique labels in batch:", len(set(y.tolist())), "(8 hona chahiye)")

train identities : 699
train images     : 57423
batches/epoch    : 87
query / gallery  : 58 / 4924
batch: (32, 3, 256, 128) | unique labels in batch: 8 (8 hona chahiye)


In [4]:
# ===== CELL 3: model =====
!pip install -q gdown yacs
!pip install -q --no-deps git+https://github.com/KaiyangZhou/deep-person-reid.git

import torch, torch.nn as nn, torch.nn.functional as F
import torchreid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARCH = "osnet_x1_0"          # baad mein "osnet_ain_x1_0" try kar sakti ho

class ReIDNet(nn.Module):
    def __init__(self, num_classes, arch=ARCH):
        super().__init__()
        self.net = torchreid.models.build_model(
            name=arch, num_classes=num_classes, loss="triplet", pretrained=True)
    def forward(self, x):
        out = self.net(x)
        if self.training:
            logits, feat = out
            return logits, feat
        return F.normalize(out, dim=1)

model = ReIDNet(NUM_CLASSES).to(device)
print("arch:", ARCH, "| params:", f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M")

x, y = next(iter(train_dl))
model.train()
logits, feat = model(x.to(device))
print("logits:", tuple(logits.shape), "| feat:", tuple(feat.shape))

  Preparing metadata (setup.py) ... done


Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 105MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
arch: osnet_x1_0 | params: 2.53M
logits: (32, 699) | feat: (32, 512)


In [5]:
# ===== CELL 4: batch-hard triplet loss =====
class BatchHardTriplet(nn.Module):
    """har anchor ke liye: sabse door positive, sabse paas negative"""
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking = nn.MarginRankingLoss(margin=margin)
    def forward(self, feat, labels):
        f = F.normalize(feat, dim=1)
        dist = torch.cdist(f, f, p=2)                      # (B, B)
        same = labels.unsqueeze(0) == labels.unsqueeze(1)  # (B, B) bool

        d_ap = (dist * same.float()).max(dim=1)[0]         # hardest positive
        d_an = (dist + same.float() * 1e6).min(dim=1)[0]   # hardest negative

        target = torch.ones_like(d_an)
        loss = self.ranking(d_an, d_ap, target)
        frac_active = (d_an - d_ap < 0.3).float().mean()   # kitne triplets abhi bhi mushkil hain
        return loss, d_ap.mean().item(), d_an.mean().item(), frac_active.item()

ce_loss  = nn.CrossEntropyLoss(label_smoothing=0.1)
tri_loss = BatchHardTriplet(margin=0.3)

# quick test
model.train()
logits, feat = model(x.to(device))
l, ap, an, act = tri_loss(feat, y.to(device))
print(f"triplet loss: {l.item():.4f} | d_ap {ap:.3f} | d_an {an:.3f} | active {act*100:.0f}%")
print("CE loss:", ce_loss(logits, y.to(device)).item())

triplet loss: 0.3501 | d_ap 1.148 | d_an 1.098 | active 100%
CE loss: 6.579993724822998


In [6]:
# ===== CELL 4b: temporal-gap eval (asli protocol) =====
import os, numpy as np
from torch.utils.data import DataLoader

per_id = {}
for path, key in split["query"] + split["gallery"]:
    per_id.setdefault(key, []).append(path)

def frame_no(p):
    try: return int(os.path.splitext(os.path.basename(p))[0])
    except ValueError: return -1

q2, g2, skipped = [], [], 0
for key, paths in per_id.items():
    paths = sorted(paths, key=frame_no)
    n = len(paths)
    if n < 8:
        skipped += 1; continue
    k = max(1, n // 4)
    q2 += [(p, key) for p in paths[:k]]
    g2 += [(p, key) for p in paths[-k:]]
print(f"hard eval -> ids: {len(per_id)-skipped} | query: {len(q2)} | gallery: {len(g2)} | skipped: {skipped}")

q_dl2 = DataLoader(EvalSet(q2, eval_tf), batch_size=64, num_workers=2)
g_dl2 = DataLoader(EvalSet(g2, eval_tf), batch_size=64, num_workers=2)

@torch.no_grad()
def extract(dl):
    model.eval()
    fs, ks = [], []
    for imgs, kk in dl:
        fs.append(model(imgs.to(device, non_blocking=True)).float().cpu())
        ks += list(kk)
    return torch.cat(fs), np.array(ks)

def eval_hard():
    qf, qk = extract(q_dl2); gf, gk = extract(g_dl2)
    sim = (qf @ gf.t()).numpy()
    cmcs, aps = [], []
    for i in range(sim.shape[0]):
        o = np.argsort(-sim[i]); m = (gk[o] == qk[i]).astype(np.int32)
        if m.sum() == 0: continue
        cmcs.append(m.cumsum().clip(max=1))
        pos = np.where(m == 1)[0]
        aps.append(float(np.mean([(j+1)/(r+1) for j, r in enumerate(pos)])))
    cmc = np.mean(cmcs, axis=0)
    return {"Rank-1": float(cmc[0]), "Rank-5": float(cmc[4]), "mAP": float(np.mean(aps))}

print("baseline (ImageNet init, training se pehle):",
      {k: round(v*100, 2) for k, v in eval_hard().items()})

hard eval -> ids: 55 | query: 1222 | gallery: 1222 | skipped: 3
baseline (ImageNet init, training se pehle): {'Rank-1': 30.11, 'Rank-5': 42.64, 'mAP': 23.82}


In [7]:
# ===== CELL 5: training =====
import time, json

EPOCHS, LR, WARMUP = 40, 3e-4, 5
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)

def lr_at(e):
    if e < WARMUP:
        return (e + 1) / WARMUP
    return 0.5 * (1 + np.cos(np.pi * (e - WARMUP) / max(1, EPOCHS - WARMUP)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

best_map, history = -1.0, []
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    tot_ce, tot_tri, tot_act, nb = 0., 0., 0., 0
    for imgs, labels in train_dl:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits, feat = model(imgs)
            l_ce = ce_loss(logits, labels)
            l_tri, _, _, act = tri_loss(feat, labels)
            loss = l_ce + l_tri
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tot_ce += l_ce.item(); tot_tri += l_tri.item(); tot_act += act; nb += 1
    scheduler.step()

    msg = (f"epoch {epoch+1:3d}/{EPOCHS} | ce {tot_ce/nb:.4f} | tri {tot_tri/nb:.4f} "
           f"| active {100*tot_act/nb:.0f}% | lr {scheduler.get_last_lr()[0]:.2e}")
    if (epoch + 1) % 2 == 0 or epoch == EPOCHS - 1:
        r = eval_hard()
        history.append({"epoch": epoch+1, **r})
        msg += f" | R1 {r['Rank-1']*100:5.2f}% mAP {r['mAP']*100:5.2f}%"
        if r["mAP"] > best_map:
            best_map = r["mAP"]
            torch.save({"state_dict": model.state_dict(), "arch": ARCH,
                        "num_classes": NUM_CLASSES, "epoch": epoch+1, "metrics": r},
                       f"{WORK}/reid_v2_best.pt")
            msg += "  <- best"
    print(msg)

print(f"\ndone in {(time.time()-t0)/60:.1f} min | best hard-protocol mAP {best_map*100:.2f}%")
json.dump(history, open(f"{WORK}/history_v2.json", "w"), indent=2)

epoch   1/40 | ce 6.7735 | tri 0.3784 | active 100% | lr 1.20e-04
epoch   2/40 | ce 6.5768 | tri 0.3659 | active 100% | lr 1.80e-04 | R1 29.87% mAP 25.74%  <- best
epoch   3/40 | ce 6.3733 | tri 0.3474 | active 100% | lr 2.40e-04
epoch   4/40 | ce 6.1280 | tri 0.3316 | active 98% | lr 3.00e-04 | R1 32.08% mAP 33.16%  <- best
epoch   5/40 | ce 5.8178 | tri 0.3154 | active 97% | lr 3.00e-04
epoch   6/40 | ce 5.4549 | tri 0.2941 | active 94% | lr 2.99e-04 | R1 37.40% mAP 37.92%  <- best
epoch   7/40 | ce 5.1302 | tri 0.2813 | active 92% | lr 2.98e-04
epoch   8/40 | ce 4.8090 | tri 0.2678 | active 88% | lr 2.95e-04 | R1 39.61% mAP 42.41%  <- best
epoch   9/40 | ce 4.5184 | tri 0.2541 | active 87% | lr 2.90e-04
epoch  10/40 | ce 4.2640 | tri 0.2467 | active 90% | lr 2.85e-04 | R1 42.96% mAP 44.20%  <- best
epoch  11/40 | ce 4.0391 | tri 0.2472 | active 87% | lr 2.79e-04
epoch  12/40 | ce 3.7981 | tri 0.2298 | active 86% | lr 2.71e-04 | R1 49.51% mAP 45.96%  <- best
epoch  13/40 | ce 3.6005 

In [8]:
# ===== CELL 6: final comparison =====
import json

ck = torch.load(f"{WORK}/reid_v2_best.pt", map_location=device, weights_only=False)
model.load_state_dict(ck["state_dict"])
r_v2 = eval_hard()

PRE = {"Rank-1": 0.2971, "Rank-5": 0.3993, "mAP": 0.2360}   # pretrained ImageNet
V1  = {"Rank-1": 0.6162, "Rank-5": 0.7561, "mAP": 0.5571}   # v1: 236 ids, random triplets

print("best epoch:", ck["epoch"], "| train identities:", NUM_CLASSES)
print("\ntemporal-gap eval (query = track ka pehla 25%, gallery = aakhri 25%)")
print("%-8s %12s %10s %10s %12s" % ("metric", "pretrained", "v1", "v2", "v1->v2"))
for k in ["Rank-1", "Rank-5", "mAP"]:
    print("%-8s %11.2f%% %9.2f%% %9.2f%% %+11.2f"
          % (k, PRE[k]*100, V1[k]*100, r_v2[k]*100, (r_v2[k]-V1[k])*100))

json.dump({"pretrained": PRE, "v1": V1, "v2": r_v2,
           "train_identities": NUM_CLASSES, "best_epoch": ck["epoch"]},
          open(f"{WORK}/hard_eval_v2.json", "w"), indent=2)
print("\nsaved:", f"{WORK}/hard_eval_v2.json")

best epoch: 36 | train identities: 699

temporal-gap eval (query = track ka pehla 25%, gallery = aakhri 25%)
metric     pretrained         v1         v2       v1->v2
Rank-1         29.71%     61.62%     66.86%       +5.24
Rank-5         39.93%     75.61%     76.60%       +0.99
mAP            23.60%     55.71%     59.21%       +3.50

saved: /kaggle/working/reid_v2/hard_eval_v2.json
